# 05 - Phân tích nhật ký thực nghiệm

Notebook này đọc `outputs/experiment_log.jsonl`, không tự tạo số liệu. Dùng để so sánh run thành công/thất bại, độ ổn định theo seed và cập nhật Bảng 4 trong báo cáo Word.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
LOG_PATH = PROJECT_ROOT / 'outputs' / 'experiment_log.jsonl'
if not LOG_PATH.exists():
    raise FileNotFoundError(f'Chua co log: {LOG_PATH}. Hay chay notebook 02 truoc.')

events = pd.DataFrame(json.loads(line) for line in LOG_PATH.read_text(encoding='utf-8').splitlines())
events['timestamp_utc'] = pd.to_datetime(events['timestamp_utc'], utc=True)
events.head()

In [ ]:
completed = events[events['event'] == 'run_completed'].copy()
failed = events[events['event'].isin(['run_failed', 'run_attempt_failed'])].copy()
print('Completed:', len(completed), 'Failed:', len(failed))

if not failed.empty:
    display(failed[['run_id', 'experiment', 'attempt', 'error_type', 'error_message']].drop_duplicates())

if not completed.empty:
    summary = completed.groupby('experiment')['best_val_macro_f1'].agg(['count', 'mean', 'std', 'max']).sort_values('mean', ascending=False)
    display(summary)
else:
    summary = pd.DataFrame(columns=['count', 'mean', 'std', 'max'])
    print('Chua co run thanh cong de tong hop.')

In [ ]:
if not summary.empty:
    summary[['mean', 'std']].plot.bar(yerr=summary['std'], capsize=4, figsize=(9, 4))
    plt.ylabel('Validation Macro-F1')
    plt.xlabel('Experiment')
    plt.title('So sanh ket qua tu experiment_log.jsonl')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'outputs' / 'log_experiment_comparison.png', dpi=160)
    plt.show()
summary.to_csv(PROJECT_ROOT / 'outputs' / 'log_summary.csv')

In [ ]:
run_table = completed[['run_id', 'experiment', 'best_val_macro_f1', 'timestamp_utc']].copy()
run_table['status'] = 'completed'

if not failed.empty:
    failed_table = failed[['run_id', 'experiment', 'attempt', 'error_type', 'timestamp_utc']].copy()
    failed_table['status'] = 'failed'
    audit_table = pd.concat([run_table, failed_table], ignore_index=True).sort_values(['run_id', 'timestamp_utc'])
else:
    audit_table = run_table.sort_values('run_id')

display(audit_table)
audit_table.to_csv(PROJECT_ROOT / 'outputs' / 'run_audit.csv', index=False)